# Rental Agreement Metadata Extraction — Demo

Walkthrough of the pipeline: load a document -> extract fields via the configured LLM -> batch predict -> compute recall.

Run this from the project root, or adjust `sys.path` below.

In [ ]:
import sys, os
sys.path.append('..')

from dotenv import load_dotenv
load_dotenv('../.env')

from app.extraction.extractor import _provider_config
_provider_config()  # raises with a clear message if the provider key is missing

## 1. Load a single document (docx example)

In [ ]:
from app.loaders import load_document_text

text = load_document_text('../data/train/18325926-Rental-Agreement-1.docx')
print(text[:800])

## 2. Extract fields with self-consistency

In [ ]:
from app.extraction import extract_fields

result = extract_fields(text)
result.to_dict()

## 3. Try a scanned image (OCR path)

In [ ]:
img_text = load_document_text('../data/train/36199312-Rental-Agreement.png')
print(img_text[:800])

img_result = extract_fields(img_text)
img_result.to_dict()

## 4. Batch predict over the full test set

In [ ]:
from app.predict import run_predictions

test_preds = run_predictions('../data/test')
test_preds

In [ ]:
os.makedirs('../outputs', exist_ok=True)
test_preds.to_csv('../outputs/test_predictions.csv', index=False)

## 5. Compute per-field recall against ground truth

In [ ]:
import pandas as pd
from app.evaluate import compute_recall

ground_truth = pd.read_csv('../data/test.csv')
scores = compute_recall(test_preds, ground_truth)
scores